In [ ]:
!pip install matplotlib==3.9.2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# خواندن داده‌ها
df = pd.read_csv('preprocessed_sales.csv')

# تبدیل ستون InvoiceDate به datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# حذف کامل مقادیر NaN و تبدیل InvoiceNumber به string
df_clean = df.dropna(subset=['InvoiceNumber']).copy()
df_clean['InvoiceNumber'] = df_clean['InvoiceNumber'].astype(str)

# قسمت اول: تعداد فاکتورهای باقی‌مانده
number_of_orders = df_clean['InvoiceNumber'].nunique()

# قسمت دوم: بازه زمانی
min_date = df_clean['InvoiceDate'].min()
max_date = df_clean['InvoiceDate'].max()
window_period = (str(min_date), str(max_date))

# قسمت سوم: نمودار روزانه
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
daily_orders = df_clean.groupby('DayOfWeek')['InvoiceNumber'].nunique()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_orders = daily_orders.reindex(days_order)

fig1, ax1 = plt.subplots(figsize=(15, 6))
bars = ax1.bar(range(len(daily_orders)), daily_orders.values, color='lime')
days_labels = ['Mon', 'Tue', 'Wed', 'Thur', 'Fri', 'Sat', 'Sun']
ax1.set_xticks(range(len(days_labels)))
ax1.set_xticklabels(days_labels, rotation=0, fontsize=15)
ax1.set_title('Number of orders for different Days', fontsize=15, color='green')
ax1.set_xlabel('Day', fontsize=15, color='lightseagreen')
ax1.set_ylabel('Number of Orders', fontsize=15, color='lightseagreen')
plt.tight_layout()

# قسمت چهارم: نمودار ماهانه
df_clean['TotalPrice'] = df_clean['UnitPrice'] * df_clean['Quantity']
df_clean['MonthYear'] = df_clean['InvoiceDate'].dt.strftime('%b_%Y')
monthly_sales = df_clean.groupby('MonthYear', sort=False)['TotalPrice'].sum()

fig2, ax2 = plt.subplots(figsize=(15, 6))
bars = ax2.bar(range(len(monthly_sales)), monthly_sales.values, color='darkkhaki')

months_labels = []
for month_year in monthly_sales.index:
    month, year = month_year.split('_')
    if month == 'Jul':
        month = 'July'
    months_labels.append(month + '_' + year)

ax2.set_xticks(range(len(months_labels)))
ax2.set_xticklabels(months_labels, rotation=45, fontsize=13)
ax2.set_title('Number of orders for different Months', fontsize=15, color='cadetblue')
ax2.set_xlabel('Month', fontsize=15, color='orange')
ax2.set_ylabel('Sales amount', fontsize=15, color='orange')
plt.tight_layout()

# نمایش نتایج
print(f"تعداد فاکتورهای یکتا: {number_of_orders}")
print(f"بازه زمانی: {window_period}")

In [ ]:
import zipfile
import joblib

joblib.dump(number_of_orders,"number_of_orders")
joblib.dump(window_period,"window_period")
joblib.dump(fig1,"fig1")
joblib.dump(fig2,"fig2")

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

file_names = ["number_of_orders", "window_period", "fig1", "fig2", "final_project_2_exploration.ipynb"]
compress(file_names)